# 6.18 - CertCF Adult Validity and Proximity Diagnostics

This notebook runs CertCF on **Adult in its normal preprocessed tabular space** and prints only aggregate quality metrics.

Goals:
- keep the setup close to the benchmark semantics
- evaluate multiple `eps_alpha` values on the same query batch
- report **validity** and **proximity** without visualization

Notes:
- training support is **prediction-aligned**, matching the benchmark stack
- the notebook uses the real Adult classifier checkpoint from the repo
- the support budget is configurable so we can trade fidelity vs runtime


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys
import time

import numpy as np
import pandas as pd
import torch
from IPython.display import display
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import TensorDataset

NOTEBOOK_DIR = Path.cwd().resolve()
ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from certcf import CertCFAtlas, NearestOppositeClassClearanceStrategy
from dataset_specs import get_tabular_dataset_spec
from models.classifiers import TabularClassifier
from training.datamodules.adult import AdultDataModule
from training.lit_classifier import LitClassifier

torch.set_grad_enabled(False)
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print({'device': str(DEVICE), 'seed': SEED})


In [ ]:
DATA_CFG = {
    'filepath': str(ROOT / 'data' / 'Adult' / 'raw.parquet'),
    'batch_size': 256,
    'val_fraction': 0.1,
    'test_fraction': 0.1,
    'seed': SEED,
    'num_workers': 0,
    'pca_enabled': False,
}

dm = AdultDataModule(**DATA_CFG)
dm.setup()
X_TRAIN, Y_TRAIN_TRUE = [t.numpy() for t in dm.train_ds.tensors]
X_TEST, Y_TEST_TRUE = [t.numpy() for t in dm.test_ds.tensors]

print({
    'train_shape': tuple(X_TRAIN.shape),
    'test_shape': tuple(X_TEST.shape),
    'train_class_counts': {int(c): int((Y_TRAIN_TRUE == c).sum()) for c in np.unique(Y_TRAIN_TRUE)},
    'test_class_counts': {int(c): int((Y_TEST_TRUE == c).sum()) for c in np.unique(Y_TEST_TRUE)},
})


In [ ]:
CKPT_PATH = ROOT / 'checkpoints' / 'adult_classifier' / 'best.ckpt'
SPEC = get_tabular_dataset_spec('adult')


def infer_tabular_classifier_dims_from_checkpoint(checkpoint: str | Path) -> tuple[list[int], int]:
    ckpt = torch.load(str(checkpoint), map_location='cpu', weights_only=False)
    state_dict = ckpt.get('state_dict', {})
    hidden_1 = state_dict.get('model.net.4.weight')
    output = state_dict.get('model.net.6.weight')
    if hidden_1 is None or output is None:
        raise KeyError('Could not infer hidden_dims / num_classes from the Adult checkpoint.')
    hidden_dims = [int(hidden_1.shape[1]), int(hidden_1.shape[0])]
    num_classes = int(output.shape[0])
    return hidden_dims, num_classes


def load_adult_model(checkpoint: str | Path, device: torch.device = DEVICE) -> torch.nn.Module:
    hidden_dims, num_classes = infer_tabular_classifier_dims_from_checkpoint(checkpoint)
    backbone = TabularClassifier(
        input_types=list(SPEC.input_types),
        cardinalities=list(SPEC.cardinalities),
        hidden_dims=hidden_dims,
        num_classes=num_classes,
        dropout=0.2,
    )
    lit = LitClassifier.load_from_checkpoint(str(checkpoint), model=backbone, map_location=str(device))
    net = lit.model.eval().to(device)
    net_no_dropout = torch.nn.Sequential(
        *[m for m in net.net if not isinstance(m, torch.nn.Dropout)]
    )
    return net_no_dropout.eval().to(device)


@torch.no_grad()
def predict_np(model: torch.nn.Module, x: np.ndarray, device: torch.device = DEVICE) -> tuple[np.ndarray, np.ndarray]:
    x_t = torch.from_numpy(np.asarray(x, dtype=np.float32)).to(device)
    logits = model(x_t)
    probs = torch.softmax(logits, dim=1).cpu().numpy().astype(np.float32)
    preds = probs.argmax(axis=1)
    return preds, probs


MODEL = load_adult_model(CKPT_PATH, device=DEVICE)
Y_TRAIN_PRED, _ = predict_np(MODEL, X_TRAIN)
Y_TEST_PRED, _ = predict_np(MODEL, X_TEST)

print({
    'checkpoint': str(CKPT_PATH),
    'predicted_train_class_counts': {int(c): int((Y_TRAIN_PRED == c).sum()) for c in np.unique(Y_TRAIN_PRED)},
    'predicted_test_class_counts': {int(c): int((Y_TEST_PRED == c).sum()) for c in np.unique(Y_TEST_PRED)},
})


## CertCF setup

The support labels are prediction-aligned, and queries target the opposite predicted class.

The support budget is configurable:
- set `support_max_per_class=None` for the full prediction-aligned support
- keep it finite for a lighter diagnostic run


In [ ]:
ALPHAS = [0.25, 0.45, 0.99]
RUN_CFG = {
    'n_queries': 200,
    'support_max_per_class': 5000,
    'batch_size': 256,
    'query_method': 'nearest_anchor',
    'query_k_candidates': 5,
    'delta': 0.0,
    'solver_maxiter': 500,
    'distance_norm': 1,
    'norm': 1,
}


def stratified_subsample(x: np.ndarray, y: np.ndarray, max_per_class: int | None, seed: int = SEED):
    if max_per_class is None:
        keep = np.arange(len(x))
        return x, y, keep
    rng = np.random.default_rng(seed)
    keep = []
    for cls in sorted(np.unique(y)):
        idx = np.where(y == cls)[0]
        if len(idx) > max_per_class:
            idx = rng.choice(idx, size=max_per_class, replace=False)
        keep.append(np.sort(idx))
    keep = np.sort(np.concatenate(keep))
    return x[keep], y[keep], keep


def choose_queries(y_pred: np.ndarray, n_queries: int, seed: int = SEED) -> np.ndarray:
    rng = np.random.default_rng(seed)
    per_class = max(1, n_queries // len(np.unique(y_pred)))
    chosen = []
    for cls in sorted(np.unique(y_pred)):
        idx = np.where(y_pred == cls)[0]
        take = min(per_class, len(idx))
        chosen.append(np.sort(rng.choice(idx, size=take, replace=False)))
    chosen = np.sort(np.concatenate(chosen))
    return chosen[:n_queries]


X_SUPPORT, Y_SUPPORT, SUPPORT_IDX = stratified_subsample(
    X_TRAIN,
    Y_TRAIN_PRED,
    max_per_class=RUN_CFG['support_max_per_class'],
)
QUERY_IDX = choose_queries(Y_TEST_PRED, RUN_CFG['n_queries'])

print({
    'support_shape': tuple(X_SUPPORT.shape),
    'support_class_counts': {int(c): int((Y_SUPPORT == c).sum()) for c in np.unique(Y_SUPPORT)},
    'n_queries_selected': int(len(QUERY_IDX)),
})


In [ ]:
def evaluate_alpha(alpha: float) -> tuple[pd.DataFrame, dict]:
    ds = TensorDataset(
        torch.from_numpy(X_SUPPORT).float(),
        torch.from_numpy(Y_SUPPORT).long(),
    )
    atlas = CertCFAtlas(
        MODEL,
        ds,
        DEVICE,
        cnn=False,
        norm=RUN_CFG['norm'],
        distance_norm=RUN_CFG['distance_norm'],
        lirpa_method='backward',
        eps_strategy=NearestOppositeClassClearanceStrategy(alpha=alpha),
        batch_size=RUN_CFG['batch_size'],
        default_query_method=RUN_CFG['query_method'],
        solver_maxiter=RUN_CFG['solver_maxiter'],
    )

    t0 = time.perf_counter()
    atlas.build(build_unions=False, verbose=True)
    build_seconds = time.perf_counter() - t0

    rows = []
    for idx in QUERY_IDX:
        x = X_TEST[idx]
        y_orig = int(Y_TEST_PRED[idx])
        target = 1 - y_orig
        q0 = time.perf_counter()
        result = atlas.find_counterfactual(
            x,
            target_class=target,
            method=RUN_CFG['query_method'],
            delta=RUN_CFG['delta'],
            query_k_candidates=RUN_CFG['query_k_candidates'],
        )
        query_seconds = time.perf_counter() - q0
        x_cf = None if result.x_cf is None else np.asarray(result.x_cf, dtype=np.float32)
        y_cf = None
        if x_cf is not None and np.all(np.isfinite(x_cf)):
            y_cf = int(predict_np(MODEL, x_cf.reshape(1, -1))[0][0])
        rows.append({
            'alpha': float(alpha),
            'query_idx': int(idx),
            'y_orig': y_orig,
            'target': target,
            'success': bool(result.success),
            'target_reached': bool(y_cf == target) if y_cf is not None else False,
            'l1': float(np.linalg.norm(x_cf - x, ord=1)) if x_cf is not None else np.nan,
            'l2': float(np.linalg.norm(x_cf - x, ord=2)) if x_cf is not None else np.nan,
            'distance_reported': float(result.distance),
            'n_qp_solved': int(result.n_qp_solved),
            'query_seconds': float(query_seconds),
        })

    detail_df = pd.DataFrame(rows)
    success_df = detail_df[detail_df['success']].copy()
    summary = {
        'alpha': float(alpha),
        'n_queries': int(len(detail_df)),
        'validity_rate': float(detail_df['success'].mean()),
        'target_reached_rate': float(detail_df['target_reached'].mean()),
        'l1_mean_success': float(success_df['l1'].mean()) if not success_df.empty else np.nan,
        'l1_median_success': float(success_df['l1'].median()) if not success_df.empty else np.nan,
        'l2_mean_success': float(success_df['l2'].mean()) if not success_df.empty else np.nan,
        'l2_median_success': float(success_df['l2'].median()) if not success_df.empty else np.nan,
        'mean_qp_solved': float(detail_df['n_qp_solved'].mean()),
        'mean_query_seconds': float(detail_df['query_seconds'].mean()),
        'build_seconds': float(build_seconds),
    }
    return detail_df, summary


detail_tables = []
summary_rows = []
for alpha in ALPHAS:
    detail_df, summary = evaluate_alpha(alpha)
    detail_tables.append(detail_df)
    summary_rows.append(summary)

DETAIL_DF = pd.concat(detail_tables, ignore_index=True)
SUMMARY_DF = pd.DataFrame(summary_rows).sort_values('alpha').reset_index(drop=True)


In [ ]:
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({
    'axes.titlesize': 11,
    'axes.labelsize': 10,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
})

display(SUMMARY_DF)

success_df = DETAIL_DF[DETAIL_DF['success']].copy()

print('\nPer-alpha successful-query proximity summary:')
display(
    success_df
    .groupby('alpha')[['l1', 'l2']]
    .agg(['mean', 'median', 'min', 'max'])
)

def _quantile_summary(series: pd.Series) -> pd.Series:
    return pd.Series({
        'min': float(series.min()),
        'p10': float(series.quantile(0.10)),
        'p25': float(series.quantile(0.25)),
        'p50': float(series.quantile(0.50)),
        'p75': float(series.quantile(0.75)),
        'p90': float(series.quantile(0.90)),
        'max': float(series.max()),
    })


print('\nL1 proximity distribution on successful queries:')
display(success_df.groupby('alpha')['l1'].apply(_quantile_summary).unstack())

print('\nL2 proximity distribution on successful queries:')
display(success_df.groupby('alpha')['l2'].apply(_quantile_summary).unstack())

plot_df = success_df.copy()
plot_df['alpha_label'] = plot_df['alpha'].map(lambda value: f'alpha={value}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, metric, title in [
    (axes[0], 'l1', 'L1 proximity distribution'),
    (axes[1], 'l2', 'L2 proximity distribution'),
]:
    drew_kde = False
    for alpha in sorted(plot_df['alpha'].unique()):
        subset = plot_df.loc[plot_df['alpha'] == alpha, metric].dropna()
        if len(subset) >= 2 and float(subset.std()) > 0.0:
            sns.kdeplot(
                data=subset,
                ax=ax,
                label=f'alpha={alpha}',
                fill=False,
                linewidth=2,
            )
            drew_kde = True
    if not drew_kde:
        sns.histplot(
            data=plot_df,
            x=metric,
            hue='alpha_label',
            element='step',
            stat='density',
            common_norm=False,
            ax=ax,
        )
    ax.set_title(title)
    ax.set_xlabel(metric.upper())
    ax.set_ylabel('Density')
    ax.legend(frameon=False)
fig.tight_layout()
plt.show()


## Questions to ask after running

- Does higher `eps_alpha` improve validity at all on Adult once support is prediction-aligned?
- If validity is already saturated, does larger `eps_alpha` still improve L1/L2 proximity?
- Are the gains large enough to justify the extra build/query cost?
- Does increasing the support budget help more than increasing `eps_alpha`?

This notebook is intentionally narrow: it is a quick CertCF-on-Adult metric notebook, not a full benchmark notebook.
